# 04 高频交易基础设施: 订单簿与撮合引擎

## 论文参考
- **作者**: Chen Yanping
- **年份**: 2024
- **标题**: *High-Frequency Trading Infrastructure Optimization*
- **关键指标**: 撮合延迟 < 1ms, 吞吐量 > 10,000 orders/sec
- **摘要**: 本文探讨高频交易的基础设施层面，包括订单簿数据结构、限价单撮合引擎、
  延迟优化等系统架构问题。不同于策略层面的研究，本文关注如何高效实现订单
  管理和撮合逻辑，是 HFT 系统的底层基石。本 notebook 实现一个简化的限价
  订单簿 (LOB) 和撮合引擎，并通过合成订单流进行性能分析。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 系统原理

### 1. 限价订单簿 (Limit Order Book)

限价订单簿维护两侧队列:

$$\text{LOB} = \{\text{Bids}(p_1^b \geq p_2^b \geq \cdots), \quad \text{Asks}(p_1^a \leq p_2^a \leq \cdots)\}$$

- **买单 (Bid)**: 按价格降序排列，最高买价优先
- **卖单 (Ask)**: 按价格升序排列，最低卖价优先

### 2. 撮合规则 (Price-Time Priority)

$$\text{Match condition}: p^{\text{bid}} \geq p^{\text{ask}}$$

- **价格优先**: 买价最高者优先成交，卖价最低者优先成交
- **时间优先**: 同价格下，先到者优先成交
- **成交价**: $p^{\text{trade}} = p^{\text{maker}}$ (挂单方价格)

### 3. 延迟模型

端到端延迟:
$$L_{\text{total}} = L_{\text{network}} + L_{\text{decode}} + L_{\text{match}} + L_{\text{risk}} + L_{\text{ack}}$$

关键优化点:
- 内存预分配避免 GC 停顿
- 无锁数据结构减少竞争
- 内核旁路 (kernel bypass) 降低网络延迟

### 4. 订单流统计

订单到达率服从泊松过程:
$$P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

买卖价差 (Bid-Ask Spread):
$$S_t = p_t^{\text{ask}} - p_t^{\text{bid}}$$

In [ ]:
# ============================================================
# Cell 3: 动画 - 订单簿深度图实时填充
# ============================================================
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# 模拟订单簿逐步堆积
mid_price = 100.0
n_steps = 40
frames = []

bid_book = {}  # price -> cumulative volume
ask_book = {}

for step in range(1, n_steps + 1):
    # 随机到达新订单
    for _ in range(np.random.randint(3, 8)):
        side = np.random.choice(['bid', 'ask'])
        if side == 'bid':
            price = round(mid_price - np.random.exponential(0.3), 2)
            price = round(price * 100) / 100  # tick size 0.01
            vol = np.random.randint(10, 200)
            bid_book[price] = bid_book.get(price, 0) + vol
        else:
            price = round(mid_price + np.random.exponential(0.3), 2)
            price = round(price * 100) / 100
            vol = np.random.randint(10, 200)
            ask_book[price] = ask_book.get(price, 0) + vol

    # 偶尔执行成交 (消耗部分挂单)
    if step % 5 == 0 and bid_book and ask_book:
        best_bid = max(bid_book.keys())
        best_ask = min(ask_book.keys())
        if best_bid >= best_ask:
            trade_vol = min(bid_book[best_bid], ask_book[best_ask]) // 2
            bid_book[best_bid] = max(0, bid_book[best_bid] - trade_vol)
            ask_book[best_ask] = max(0, ask_book[best_ask] - trade_vol)
            if bid_book[best_bid] <= 0: del bid_book[best_bid]
            if ask_book[best_ask] <= 0: del ask_book[best_ask]

    # 缓慢漂移mid price
    mid_price += np.random.randn() * 0.02

    # 构建深度图数据 (累积量)
    if bid_book:
        bid_prices = sorted(bid_book.keys(), reverse=True)
        bid_cum_vol = np.cumsum([bid_book[p] for p in bid_prices])
    else:
        bid_prices, bid_cum_vol = [], []

    if ask_book:
        ask_prices = sorted(ask_book.keys())
        ask_cum_vol = np.cumsum([ask_book[p] for p in ask_prices])
    else:
        ask_prices, ask_cum_vol = [], []

    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(bid_prices), y=list(bid_cum_vol),
                       fill='tozeroy', mode='lines',
                       fillcolor='rgba(76, 175, 80, 0.3)',
                       line=dict(color='#4CAF50', width=2),
                       name='Bid (Buy)'),
            go.Scatter(x=list(ask_prices), y=list(ask_cum_vol),
                       fill='tozeroy', mode='lines',
                       fillcolor='rgba(244, 67, 54, 0.3)',
                       line=dict(color='#F44336', width=2),
                       name='Ask (Sell)'),
        ],
        name=f'Step {step}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='Order Book Depth Chart: Real-time Fill Animation'),
        xaxis_title='Price',
        yaxis_title='Cumulative Volume',
        height=500, width=900,
        xaxis=dict(range=[mid_price - 2, mid_price + 2]),
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 Play', method='animate',
                 args=[None, {'frame': {'duration': 300}, 'transition': {'duration': 150}}]),
            dict(label='\u23f8 Pause', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与配置
# ============================================================
import sys
import time
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
import heapq

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.visualization import set_chinese_font
from shared.constants import COLOR_BUY, COLOR_SELL, COLOR_STRATEGY

print('All imports successful.')

In [ ]:
# ============================================================
# Cell 5: 合成订单流生成器
# ============================================================

@dataclass
class Order:
    """限价单数据结构"""
    order_id: int
    timestamp: float       # 纳秒时间戳
    side: str              # 'buy' or 'sell'
    price: float           # 限价
    quantity: int          # 数量
    remaining: int = 0     # 剩余数量

    def __post_init__(self):
        self.remaining = self.quantity


@dataclass
class Trade:
    """成交记录"""
    trade_id: int
    timestamp: float
    price: float
    quantity: int
    buyer_order_id: int
    seller_order_id: int


def generate_order_flow(n_orders: int = 5000, mid_price: float = 100.0,
                         tick_size: float = 0.01,
                         lambda_rate: float = 100.0) -> List[Order]:
    """
    生成合成限价订单流

    参数:
        n_orders: 订单总数
        mid_price: 初始中间价
        tick_size: 最小价格变动
        lambda_rate: 泊松到达率 (orders/sec)
    """
    orders = []
    current_time = 0.0
    price = mid_price

    for i in range(n_orders):
        # 泊松过程间隔
        inter_arrival = np.random.exponential(1.0 / lambda_rate)
        current_time += inter_arrival

        # 随机买卖方向
        side = np.random.choice(['buy', 'sell'])

        # 价格: 围绕mid_price的指数分布偏移
        if side == 'buy':
            offset = np.random.exponential(0.1)
            order_price = round((price - offset) / tick_size) * tick_size
        else:
            offset = np.random.exponential(0.1)
            order_price = round((price + offset) / tick_size) * tick_size

        # 偶尔市价单 (穿越spread)
        if np.random.random() < 0.15:
            if side == 'buy':
                order_price = price + np.random.uniform(0, 0.2)
            else:
                order_price = price - np.random.uniform(0, 0.2)
            order_price = round(order_price / tick_size) * tick_size

        # 数量: 对数正态分布
        qty = max(1, int(np.random.lognormal(4, 1)))

        orders.append(Order(
            order_id=i,
            timestamp=current_time,
            side=side,
            price=max(order_price, tick_size),
            quantity=qty,
        ))

        # 缓慢随机游走
        price += np.random.randn() * 0.005

    print(f'Generated {n_orders} synthetic orders')
    print(f'Time span: {orders[0].timestamp:.2f}s ~ {orders[-1].timestamp:.2f}s')
    buy_count = sum(1 for o in orders if o.side == 'buy')
    print(f'Buy orders: {buy_count}, Sell orders: {n_orders - buy_count}')
    return orders


orders = generate_order_flow(n_orders=10000, mid_price=100.0)

In [ ]:
# ============================================================
# Cell 6: 限价订单簿 + 撮合引擎实现
# ============================================================

class OrderBook:
    """
    简化限价订单簿 (LOB) 和撮合引擎

    数据结构:
      - bids: max-heap (negate price for max behavior with heapq)
      - asks: min-heap
      - price_levels: {price: [Order, ...]} 每个价位的挂单队列

    撮合规则: Price-Time Priority
    """

    def __init__(self):
        self.bids = []          # max-heap: (-price, timestamp, order_id)
        self.asks = []          # min-heap: (price, timestamp, order_id)
        self.orders = {}        # order_id -> Order
        self.trades = []        # 成交记录
        self.trade_counter = 0

        # 统计
        self.latencies = []     # 每笔撮合延迟 (秒)
        self.n_matched = 0
        self.total_volume = 0
        self.spread_history = []
        self.mid_price_history = []

    def add_order(self, order: Order) -> List[Trade]:
        """添加订单并尝试撮合"""
        t_start = time.perf_counter_ns()

        new_trades = []
        self.orders[order.order_id] = order

        if order.side == 'buy':
            new_trades = self._match_buy(order)
            if order.remaining > 0:
                heapq.heappush(self.bids,
                    (-order.price, order.timestamp, order.order_id))
        else:
            new_trades = self._match_sell(order)
            if order.remaining > 0:
                heapq.heappush(self.asks,
                    (order.price, order.timestamp, order.order_id))

        t_end = time.perf_counter_ns()
        latency_us = (t_end - t_start) / 1000  # microseconds
        self.latencies.append(latency_us)

        # 记录spread和mid
        best_bid = self.best_bid()
        best_ask = self.best_ask()
        if best_bid and best_ask:
            self.spread_history.append(best_ask - best_bid)
            self.mid_price_history.append((best_bid + best_ask) / 2)

        return new_trades

    def _match_buy(self, buy_order: Order) -> List[Trade]:
        """买单撮合: 与asks中价格<=买价的卖单成交"""
        trades = []
        while buy_order.remaining > 0 and self.asks:
            ask_price, ask_ts, ask_id = self.asks[0]
            if ask_price > buy_order.price:
                break  # 没有可匹配的卖单

            sell_order = self.orders.get(ask_id)
            if sell_order is None or sell_order.remaining <= 0:
                heapq.heappop(self.asks)
                continue

            # 成交
            fill_qty = min(buy_order.remaining, sell_order.remaining)
            trade_price = ask_price  # maker price

            trade = Trade(
                trade_id=self.trade_counter,
                timestamp=buy_order.timestamp,
                price=trade_price,
                quantity=fill_qty,
                buyer_order_id=buy_order.order_id,
                seller_order_id=ask_id,
            )
            trades.append(trade)
            self.trades.append(trade)
            self.trade_counter += 1
            self.n_matched += 1
            self.total_volume += fill_qty

            buy_order.remaining -= fill_qty
            sell_order.remaining -= fill_qty

            if sell_order.remaining <= 0:
                heapq.heappop(self.asks)

        return trades

    def _match_sell(self, sell_order: Order) -> List[Trade]:
        """卖单撮合: 与bids中价格>=卖价的买单成交"""
        trades = []
        while sell_order.remaining > 0 and self.bids:
            neg_bid_price, bid_ts, bid_id = self.bids[0]
            bid_price = -neg_bid_price
            if bid_price < sell_order.price:
                break

            buy_order = self.orders.get(bid_id)
            if buy_order is None or buy_order.remaining <= 0:
                heapq.heappop(self.bids)
                continue

            fill_qty = min(sell_order.remaining, buy_order.remaining)
            trade_price = bid_price  # maker price

            trade = Trade(
                trade_id=self.trade_counter,
                timestamp=sell_order.timestamp,
                price=trade_price,
                quantity=fill_qty,
                buyer_order_id=bid_id,
                seller_order_id=sell_order.order_id,
            )
            trades.append(trade)
            self.trades.append(trade)
            self.trade_counter += 1
            self.n_matched += 1
            self.total_volume += fill_qty

            sell_order.remaining -= fill_qty
            buy_order.remaining -= fill_qty

            if buy_order.remaining <= 0:
                heapq.heappop(self.bids)

        return trades

    def best_bid(self) -> Optional[float]:
        """最优买价"""
        while self.bids:
            neg_p, ts, oid = self.bids[0]
            order = self.orders.get(oid)
            if order and order.remaining > 0:
                return -neg_p
            heapq.heappop(self.bids)
        return None

    def best_ask(self) -> Optional[float]:
        """最优卖价"""
        while self.asks:
            p, ts, oid = self.asks[0]
            order = self.orders.get(oid)
            if order and order.remaining > 0:
                return p
            heapq.heappop(self.asks)
        return None

    def depth(self, n_levels: int = 5) -> dict:
        """获取n档深度"""
        bid_levels = defaultdict(int)
        for neg_p, ts, oid in self.bids:
            order = self.orders.get(oid)
            if order and order.remaining > 0:
                bid_levels[-neg_p] += order.remaining
        ask_levels = defaultdict(int)
        for p, ts, oid in self.asks:
            order = self.orders.get(oid)
            if order and order.remaining > 0:
                ask_levels[p] += order.remaining

        top_bids = sorted(bid_levels.items(), reverse=True)[:n_levels]
        top_asks = sorted(ask_levels.items())[:n_levels]
        return {'bids': top_bids, 'asks': top_asks}


print('OrderBook matching engine defined.')

In [ ]:
# ============================================================
# Cell 7: 运行撮合引擎 + 性能测量
# ============================================================

ob = OrderBook()

# 逐笔处理订单
wall_start = time.perf_counter()

for order in orders:
    ob.add_order(order)

wall_end = time.perf_counter()
wall_time = wall_end - wall_start

# 统计
n_orders = len(orders)
n_trades = len(ob.trades)
throughput = n_orders / wall_time

latencies_us = np.array(ob.latencies)
lat_mean = np.mean(latencies_us)
lat_median = np.median(latencies_us)
lat_p99 = np.percentile(latencies_us, 99)
lat_max = np.max(latencies_us)

print(f'=== Matching Engine Performance ===')
print(f'  Orders processed: {n_orders:,}')
print(f'  Trades executed:  {n_trades:,}')
print(f'  Total volume:     {ob.total_volume:,}')
print(f'  Fill rate:        {n_trades / n_orders:.1%}')
print(f'')
print(f'=== Latency Statistics (microseconds) ===')
print(f'  Mean:   {lat_mean:.1f} us')
print(f'  Median: {lat_median:.1f} us')
print(f'  P99:    {lat_p99:.1f} us')
print(f'  Max:    {lat_max:.1f} us')
print(f'')
print(f'=== Throughput ===')
print(f'  Wall time:  {wall_time:.3f}s')
print(f'  Throughput: {throughput:,.0f} orders/sec')

# 深度快照
depth = ob.depth(5)
print(f'\n=== Final Order Book (Top 5) ===')
print(f'  Asks:')
for price, vol in reversed(depth['asks']):
    print(f'    {price:.2f} x {vol}')
print(f'  ---')
print(f'  Bids:')
for price, vol in depth['bids']:
    print(f'    {price:.2f} x {vol}')

In [ ]:
# ============================================================
# Cell 8: 订单流特征分析
# ============================================================

# 分析成交数据
trade_prices = [t.price for t in ob.trades]
trade_volumes = [t.quantity for t in ob.trades]
trade_times = [t.timestamp for t in ob.trades]

# 订单间隔
inter_arrivals = np.diff([o.timestamp for o in orders])

# Spread 统计
spreads = np.array(ob.spread_history)
spreads_valid = spreads[spreads > 0]

print(f'=== Trade Analysis ===')
if trade_prices:
    print(f'  Price range:  {min(trade_prices):.2f} - {max(trade_prices):.2f}')
    print(f'  Avg volume:   {np.mean(trade_volumes):.0f}')
    print(f'  Volume std:   {np.std(trade_volumes):.0f}')

print(f'\n=== Order Arrival ===')
print(f'  Mean interval: {np.mean(inter_arrivals)*1000:.2f} ms')
print(f'  Std interval:  {np.std(inter_arrivals)*1000:.2f} ms')

print(f'\n=== Bid-Ask Spread ===')
if len(spreads_valid) > 0:
    print(f'  Mean spread:   {np.mean(spreads_valid):.4f}')
    print(f'  Median spread: {np.median(spreads_valid):.4f}')
    print(f'  Min spread:    {np.min(spreads_valid):.4f}')
    print(f'  Max spread:    {np.max(spreads_valid):.4f}')

print(f'\n=== Mid-Price ===')
mid_prices = np.array(ob.mid_price_history)
if len(mid_prices) > 0:
    print(f'  Start: {mid_prices[0]:.4f}')
    print(f'  End:   {mid_prices[-1]:.4f}')
    print(f'  Drift: {mid_prices[-1] - mid_prices[0]:.4f}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Latency Distribution
ax = axes[0, 0]
ax.hist(latencies_us, bins=80, color=COLOR_STRATEGY, alpha=0.7, edgecolor='white')
ax.axvline(lat_mean, color='red', linestyle='--', label=f'Mean={lat_mean:.1f}us')
ax.axvline(lat_p99, color='orange', linestyle='--', label=f'P99={lat_p99:.1f}us')
ax.set_title('Matching Latency Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Latency (microseconds)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Mid-Price Evolution
ax = axes[0, 1]
if len(mid_prices) > 0:
    ax.plot(mid_prices, color='#333333', linewidth=0.5)
    ax.set_title('Mid-Price Over Time', fontsize=12, fontweight='bold')
    ax.set_xlabel('Order Sequence')
    ax.set_ylabel('Mid Price')
    ax.grid(True, alpha=0.3)

# 3. Bid-Ask Spread
ax = axes[0, 2]
if len(spreads_valid) > 0:
    ax.plot(spreads_valid[:2000], color='#FF9800', linewidth=0.5, alpha=0.7)
    spread_ma = pd.Series(spreads_valid).rolling(50).mean()
    ax.plot(spread_ma.values[:2000], color='#F44336', linewidth=2, label='MA50')
    ax.set_title('Bid-Ask Spread', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Spread')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# 4. Trade Volume Distribution
ax = axes[1, 0]
if trade_volumes:
    ax.hist(trade_volumes, bins=50, color='#4CAF50', alpha=0.7, edgecolor='white')
    ax.set_title('Trade Volume Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel('Volume per Trade')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

# 5. Inter-Arrival Times
ax = axes[1, 1]
ax.hist(inter_arrivals * 1000, bins=80, color='#9C27B0', alpha=0.7, edgecolor='white')
ax.set_title('Order Inter-Arrival Times', fontsize=12, fontweight='bold')
ax.set_xlabel('Inter-Arrival (ms)')
ax.set_ylabel('Count')
ax.grid(True, alpha=0.3)

# 6. Cumulative Throughput
ax = axes[1, 2]
order_times = np.array([o.timestamp for o in orders])
# Orders per second in sliding window
window_sec = 1.0
time_bins = np.arange(order_times[0], order_times[-1], window_sec)
throughputs = []
for t in time_bins:
    count = np.sum((order_times >= t) & (order_times < t + window_sec))
    throughputs.append(count)
ax.plot(time_bins, throughputs, color='#00BCD4', linewidth=1)
ax.axhline(np.mean(throughputs), color='red', linestyle='--',
           label=f'Avg={np.mean(throughputs):.0f}/s')
ax.set_title('Throughput (Orders/sec)', fontsize=12, fontweight='bold')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Orders per Second')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
from shared.visualization import plot_metrics_table

infra_metrics = {
    'Total Orders': f'{n_orders:,}',
    'Total Trades': f'{n_trades:,}',
    'Fill Rate': f'{n_trades / n_orders:.1%}',
    'Total Volume': f'{ob.total_volume:,}',
    'Mean Latency': f'{lat_mean:.1f} us',
    'P99 Latency': f'{lat_p99:.1f} us',
    'Max Latency': f'{lat_max:.1f} us',
    'Throughput': f'{throughput:,.0f} orders/sec',
    'Mean Spread': f'{np.mean(spreads_valid):.4f}' if len(spreads_valid) > 0 else 'N/A',
    'Avg Trade Volume': f'{np.mean(trade_volumes):.0f}' if trade_volumes else 'N/A',
}
plot_metrics_table(infra_metrics, title='HFT Infrastructure - Performance Metrics')

## 结果讨论

### 系统表现

1. **撮合延迟**: Python 实现的单笔撮合在微秒级别，生产系统 (C++) 通常在百纳秒级
2. **吞吐量**: 纯 Python 可达数千至万单/秒，C++ 实现通常 > 100万单/秒
3. **订单簿深度**: 合成订单流形成了合理的价格分层和流动性分布

### 关键系统概念

- **Price-Time Priority**: 最基础的撮合规则，保证公平性
- **Heap 数据结构**: 用最小堆管理卖单、最大堆管理买单，保证 O(log N) 复杂度
- **Bid-Ask Spread**: 反映市场流动性，窄 spread 意味着流动性好

### 与 Chen (2024) 对比

- Chen 讨论了内核旁路、FPGA 加速、无锁队列等硬件级优化
- 本实现为纯 Python 教学版本，重在理解核心逻辑
- 生产 HFT 系统还需考虑: 网络共置 (co-location)、硬件时间戳、风控前置

### 改进方向

- 实现撤单 (Cancel) 和改单 (Amend) 功能
- 添加冰山单 (Iceberg Order) 和隐藏单支持
- 使用 Cython/C++ 重写核心撮合路径
- 引入市场冲击模型 (Market Impact) 进行更真实的模拟
- 多资产订单簿和跨市场套利模拟